# Hybrid Comparison Across IDM Priors

In [20]:
# ============================================================
# IMPORTS
# ============================================================

import copy
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
    classification_report,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("Imports OK")

Imports OK


In [21]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("SEED:", SEED)
print("DEVICE:", DEVICE)

SEED: 42
DEVICE: cuda


In [22]:
# ============================================================
# CONFIG
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility"
)

DATA_DIR = PROJECT_DIR / "processed_data_final"

OUTPUT_DIR = DATA_DIR / "hybrid_idm_first_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print("Device:", DEVICE)

assert DATA_DIR.exists(), f"DATA_DIR does not exist: {DATA_DIR}"

Data dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final
Output dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\hybrid_idm_first_runs
Device: cuda


In [23]:
# ============================================================
# LOAD PRE-SPLIT DATASETS
# ============================================================

required_files = [
    DATA_DIR / "episode_tracks_train.csv",
    DATA_DIR / "episode_tracks_val.csv",
    DATA_DIR / "episode_tracks_test.csv",
]

for p in required_files:
    assert p.exists(), f"Missing required file: {p}"

episode_tracks_train = pd.read_csv(DATA_DIR / "episode_tracks_train.csv")
episode_tracks_val   = pd.read_csv(DATA_DIR / "episode_tracks_val.csv")
episode_tracks_test  = pd.read_csv(DATA_DIR / "episode_tracks_test.csv")

print("\n=== SPLIT CHECKS ===")

for name, df in [
    ("train", episode_tracks_train),
    ("val", episode_tracks_val),
    ("test", episode_tracks_test),
]:
    print(f"\n{name}:")
    print("rows:", len(df))
    print("egos:", df["unique_id"].nunique())
    print("episodes:", df["episode_id"].nunique())
    print("split values:")
    print(df["split"].value_counts())

all_loaded = pd.concat(
    [episode_tracks_train, episode_tracks_val, episode_tracks_test],
    ignore_index=True
)

ego_split_check = all_loaded.groupby("unique_id")["split"].nunique()
episode_split_check = all_loaded.groupby("episode_id")["split"].nunique()

print("\nEgos in multiple splits:", (ego_split_check > 1).sum())
print("Episodes in multiple splits:", (episode_split_check > 1).sum())

assert (ego_split_check <= 1).all(), "Some egos appear in multiple splits."
assert (episode_split_check <= 1).all(), "Some episodes appear in multiple splits."

print("\nSplit integrity checks passed.")


=== SPLIT CHECKS ===

train:
rows: 11094352
egos: 34862
episodes: 34870
split values:
split
train    11094352
Name: count, dtype: int64

val:
rows: 2370107
egos: 7470
episodes: 7470
split values:
split
val    2370107
Name: count, dtype: int64

test:
rows: 2368174
egos: 7471
episodes: 7471
split values:
split
test    2368174
Name: count, dtype: int64

Egos in multiple splits: 0
Episodes in multiple splits: 0

Split integrity checks passed.


In [24]:
# ============================================================
# KEEP REQUIRED COLUMNS
# ============================================================

keep_cols = [
    "recording_id",
    "unique_id",
    "frame",
    "id",
    "precedingId",
    "preceding_unique_id",
    "laneId",
    "xVelocity",
    "precedingXVelocity",
    "dhw",
    "xAcceleration",
    "episode_id",
    "split"
]

for col in keep_cols:
    assert col in episode_tracks_train.columns, f"Missing column in train: {col}"
    assert col in episode_tracks_val.columns, f"Missing column in val: {col}"
    assert col in episode_tracks_test.columns, f"Missing column in test: {col}"

train_df_raw = episode_tracks_train[keep_cols].copy()
val_df_raw   = episode_tracks_val[keep_cols].copy()
test_df_raw  = episode_tracks_test[keep_cols].copy()

print("train:", train_df_raw.shape)
print("val  :", val_df_raw.shape)
print("test :", test_df_raw.shape)

train: (11094352, 13)
val  : (2370107, 13)
test : (2368174, 13)


In [25]:
# ============================================================
# FINAL IDM PARAMETER SETS
# ============================================================

best_global_params = {"v0": 40.0, "T": 0.5,"s0": 0.5, "a": 0.4,"b": 5.0,"delta": 4.0}

best_params_braking = {'v0': 1.5, 'T': 0.3, 's0': 5.0, 'a': 0.1, 'b': 26.0, 'delta': 0.5}

best_params_accel = {'v0': 115.0, 'T': 0.1, 's0': 0.05, 'a': 0.4, 'b': 22.0, 'delta': 14.0}

best_params_steady = {'v0': 44.0, 'T': 0.3, 's0': 0.1, 'a': 0.1, 'b': 8.0, 'delta': 2.0}


In [26]:
# ============================================================
# REGIME THRESHOLD CONFIGURATION
# ============================================================

BRAKE_THRESHOLD = -0.2
ACCEL_THRESHOLD = 0.2

IDM_PARAM_TAG = "refinedIDM_v2"

THRESHOLD_TAG = f"brake{abs(BRAKE_THRESHOLD):.1f}_accel{ACCEL_THRESHOLD:.1f}".replace(".", "p")

print("Using regime definition:")
print(f"braking      : a_true < {BRAKE_THRESHOLD}")
print(f"accelerating : a_true > {ACCEL_THRESHOLD}")
print("steady       : otherwise")
print("Threshold tag:", THRESHOLD_TAG)

Using regime definition:
braking      : a_true < -0.2
accelerating : a_true > 0.2
steady       : otherwise
Threshold tag: brake0p2_accel0p2


In [27]:
# ============================================================
# IDM + FEATURE + METRIC HELPERS
# ============================================================

def idm_acceleration(v, delta_v, s, params):
    v0 = max(float(params["v0"]), 1e-6)
    T = float(params["T"])
    a = max(float(params["a"]), 1e-6)
    b = max(float(params["b"]), 1e-6)
    s0 = float(params["s0"])
    delta = float(params["delta"])

    v = np.maximum(np.asarray(v), 0.0)
    s = np.maximum(np.asarray(s), 0.1)

    sqrt_ab = np.sqrt(a * b)

    s_star = s0 + np.maximum(
        0.0,
        v * T + (v * delta_v) / (2.0 * sqrt_ab)
    )

    acc = a * (1.0 - (v / v0) ** delta - (s_star / s) ** 2)
    return acc


def prepare_idm_features(df):
    out = df.copy()

    out["v_ego"] = out["xVelocity"].abs()
    out["v_leader"] = out["precedingXVelocity"].abs()
    out["a_true"] = out["xAcceleration"]

    no_leader = (
        out["precedingId"].isna()
        | (out["precedingId"] <= 0)
        | out["precedingXVelocity"].isna()
        | out["dhw"].isna()
    )

    out["delta_v"] = out["v_ego"] - out["v_leader"]
    out["gap"] = out["dhw"].clip(lower=0.1)

    out.loc[no_leader, "gap"] = 1e6
    out.loc[no_leader, "delta_v"] = 0.0
    out.loc[no_leader, "v_leader"] = out.loc[no_leader, "v_ego"]

    out["no_leader"] = no_leader.astype(int)

    return out


def add_selector_features(df):
    out = df.copy()

    out["time_headway"] = out["gap"] / np.maximum(out["v_ego"], 0.1)
    out["closing_speed"] = np.maximum(out["delta_v"], 0.0)
    out["opening_speed"] = np.maximum(-out["delta_v"], 0.0)
    out["abs_delta_v"] = np.abs(out["delta_v"])
    out["inv_ttc"] = out["closing_speed"] / np.maximum(out["gap"], 0.1)
    out["log_gap"] = np.log1p(np.minimum(out["gap"], 500.0))

    return out


def add_regime_column(df, true_col="a_true"):
    out = df.copy()

    out["regime"] = np.where(
        out[true_col] < BRAKE_THRESHOLD,
        "braking",
        np.where(
            out[true_col] > ACCEL_THRESHOLD,
            "accelerating",
            "steady"
        )
    )

    return out


def filter_regime(df, regime, true_col="xAcceleration"):
    if regime == "braking":
        return df[df[true_col] < BRAKE_THRESHOLD].copy()
    elif regime == "accelerating":
        return df[df[true_col] > ACCEL_THRESHOLD].copy()
    elif regime == "steady":
        return df[
            (df[true_col] >= BRAKE_THRESHOLD)
            & (df[true_col] <= ACCEL_THRESHOLD)
        ].copy()
    else:
        raise ValueError("regime must be 'braking', 'accelerating', or 'steady'")


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    corr = np.nan
    if len(y_true) > 1 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        corr = float(np.corrcoef(y_true, y_pred)[0, 1])

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MedAE": median_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Bias": float(np.mean(y_pred - y_true)),
        "Corr": corr,
    }


def compute_regime_metrics_from_arrays(y_true, y_pred):
    temp = pd.DataFrame({"a_true": y_true, "a_pred": y_pred})
    temp = add_regime_column(temp, true_col="a_true")

    regime_metrics = (
        temp.groupby("regime", as_index=False)
        .apply(lambda g: pd.Series({
            "MAE": np.mean(np.abs(g["a_true"] - g["a_pred"])),
            "RMSE": np.sqrt(np.mean((g["a_true"] - g["a_pred"]) ** 2)),
            "Bias": np.mean(g["a_pred"] - g["a_true"]),
            "Count": len(g)
        }))
        .reset_index(drop=True)
    )

    regime_order = ["accelerating", "braking", "steady"]
    regime_metrics["regime"] = pd.Categorical(
        regime_metrics["regime"],
        categories=regime_order,
        ordered=True
    )

    return regime_metrics.sort_values("regime").reset_index(drop=True)

In [28]:
# ============================================================
# BUILD TRUE-REGIME SELECTOR DATASETS
# ============================================================

selector_feature_cols = [
    "v_ego",
    "v_leader",
    "delta_v",
    "gap",
    "time_headway",
    "closing_speed",
    "opening_speed",
    "abs_delta_v",
    "inv_ttc",
    "log_gap"
]

selector_target_col = "regime"


def build_regime_dataset(df):
    out = prepare_idm_features(df)
    out = add_selector_features(out)
    out = add_regime_column(out, true_col="a_true")

    out["a_idm_global"] = idm_acceleration(
        v=out["v_ego"],
        delta_v=out["delta_v"],
        s=out["gap"],
        params=best_global_params
    ).clip(-8, 4)

    return out


train_regime_df = build_regime_dataset(train_df_raw)
val_regime_df   = build_regime_dataset(val_df_raw)
test_regime_df  = build_regime_dataset(test_df_raw)

print("Selector features:")
print(selector_feature_cols)

print("\nTrain regime counts:")
print(train_regime_df["regime"].value_counts())

print("\nVal regime counts:")
print(val_regime_df["regime"].value_counts())

print("\nTest regime counts:")
print(test_regime_df["regime"].value_counts())

Selector features:
['v_ego', 'v_leader', 'delta_v', 'gap', 'time_headway', 'closing_speed', 'opening_speed', 'abs_delta_v', 'inv_ttc', 'log_gap']

Train regime counts:
regime
steady          6522921
accelerating    3182947
braking         1388484
Name: count, dtype: int64

Val regime counts:
regime
steady          1385845
accelerating     685291
braking          298971
Name: count, dtype: int64

Test regime counts:
regime
steady          1411748
accelerating     672329
braking          284097
Name: count, dtype: int64


In [29]:
# ============================================================
# TRAIN TRUE-REGIME SELECTORS
# ============================================================

X_train_sel = train_regime_df[selector_feature_cols]
y_train_sel = train_regime_df[selector_target_col]

X_val_sel = val_regime_df[selector_feature_cols]
y_val_sel = val_regime_df[selector_target_col]

selector_tree = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=500,
    random_state=SEED,
    class_weight="balanced"
)

selector_tree.fit(X_train_sel, y_train_sel)

selector_gnb = GaussianNB()
selector_gnb.fit(X_train_sel, y_train_sel)

print("=== DecisionTree validation report ===")
print(classification_report(y_val_sel, selector_tree.predict(X_val_sel), digits=4))

print("=== GaussianNB validation report ===")
print(classification_report(y_val_sel, selector_gnb.predict(X_val_sel), digits=4))

=== DecisionTree validation report ===
              precision    recall  f1-score   support

accelerating     0.3964    0.5162    0.4485    685291
     braking     0.2337    0.6168    0.3390    298971
      steady     0.7889    0.3920    0.5237   1385845

    accuracy                         0.4563   2370107
   macro avg     0.4730    0.5083    0.4370   2370107
weighted avg     0.6054    0.4563    0.4787   2370107

=== GaussianNB validation report ===
              precision    recall  f1-score   support

accelerating     0.4125    0.1891    0.2593    685291
     braking     0.2896    0.3344    0.3104    298971
      steady     0.6492    0.8014    0.7173   1385845

    accuracy                         0.5654   2370107
   macro avg     0.4504    0.4416    0.4290   2370107
weighted avg     0.5354    0.5654    0.5335   2370107



In [30]:
# ============================================================
# XGB-SOFT BEST-IDM SELECTOR
# Trains a feasible selector to predict the best IDM candidate.
# The final IDM prior is a probability-weighted soft mixture.
# ============================================================

def add_all_idm_candidates_for_selector(df):
    out = prepare_idm_features(df)
    out = add_selector_features(out)

    out["a_idm_global"] = idm_acceleration(
        v=out["v_ego"],
        delta_v=out["delta_v"],
        s=out["gap"],
        params=best_global_params
    ).clip(-8, 4)

    out["a_idm_braking"] = idm_acceleration(
        v=out["v_ego"],
        delta_v=out["delta_v"],
        s=out["gap"],
        params=best_params_braking
    ).clip(-8, 4)

    out["a_idm_accel"] = idm_acceleration(
        v=out["v_ego"],
        delta_v=out["delta_v"],
        s=out["gap"],
        params=best_params_accel
    ).clip(-8, 4)

    out["a_idm_steady"] = idm_acceleration(
        v=out["v_ego"],
        delta_v=out["delta_v"],
        s=out["gap"],
        params=best_params_steady
    ).clip(-8, 4)

    return out


def add_best_idm_label(df):
    out = df.copy()

    candidate_cols = [
        "a_idm_global",
        "a_idm_braking",
        "a_idm_accel",
        "a_idm_steady"
    ]

    errors = np.column_stack([
        np.abs(out["a_true"].values - out[col].values)
        for col in candidate_cols
    ])

    best_idx = np.argmin(errors, axis=1)

    label_names = np.array([
        "global",
        "braking",
        "accelerating",
        "steady"
    ])

    out["best_idm_label"] = label_names[best_idx]
    out["best_idm_error"] = errors[np.arange(len(out)), best_idx]

    return out


def build_best_idm_selector_df(df):
    out = add_all_idm_candidates_for_selector(df)
    out = add_best_idm_label(out)
    return out


train_best_idm_df = build_best_idm_selector_df(train_df_raw)
val_best_idm_df   = build_best_idm_selector_df(val_df_raw)
test_best_idm_df  = build_best_idm_selector_df(test_df_raw)

le_best = LabelEncoder()

y_train_xgb_best = le_best.fit_transform(train_best_idm_df["best_idm_label"])
y_val_xgb_best   = le_best.transform(val_best_idm_df["best_idm_label"])

xgb_best_selector = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist"
)

xgb_best_selector.fit(
    train_best_idm_df[selector_feature_cols],
    y_train_xgb_best
)

val_pred_idx = xgb_best_selector.predict(val_best_idm_df[selector_feature_cols])
val_pred_label = le_best.inverse_transform(val_pred_idx.astype(int))

print("=== XGB BEST-IDM SELECTOR VALIDATION REPORT ===")
print(classification_report(
    val_best_idm_df["best_idm_label"],
    val_pred_label,
    digits=4
))

print("XGB best-IDM classes:")
print(le_best.classes_)

xgb_best_selector_bundle = {
    "model": xgb_best_selector,
    "label_encoder": le_best
}

=== XGB BEST-IDM SELECTOR VALIDATION REPORT ===
              precision    recall  f1-score   support

accelerating     0.4809    0.5299    0.5042    496232
     braking     0.5605    0.3217    0.4088    369409
      global     0.3693    0.1255    0.1874    454273
      steady     0.5566    0.7722    0.6469   1050193

    accuracy                         0.5273   2370107
   macro avg     0.4918    0.4373    0.4368   2370107
weighted avg     0.5055    0.5273    0.4918   2370107

XGB best-IDM classes:
['accelerating' 'braking' 'global' 'steady']


In [31]:
# ============================================================
# BUILD IDM PRIOR DATAFRAME
# ============================================================

def build_prior_dataframe(df, prior_mode, selector=None):
    out = prepare_idm_features(df)
    out = add_selector_features(out)

    # -------------------------------------------------
    # Candidate IDM priors
    # -------------------------------------------------
    out["a_idm_global_raw"] = idm_acceleration(
        v=out["v_ego"],
        delta_v=out["delta_v"],
        s=out["gap"],
        params=best_global_params
    )

    out["a_idm_global"] = out["a_idm_global_raw"].clip(-8, 4)

    out["a_idm_braking"] = idm_acceleration(
        v=out["v_ego"],
        delta_v=out["delta_v"],
        s=out["gap"],
        params=best_params_braking
    ).clip(-8, 4)

    out["a_idm_accel"] = idm_acceleration(
        v=out["v_ego"],
        delta_v=out["delta_v"],
        s=out["gap"],
        params=best_params_accel
    ).clip(-8, 4)

    out["a_idm_steady"] = idm_acceleration(
        v=out["v_ego"],
        delta_v=out["delta_v"],
        s=out["gap"],
        params=best_params_steady
    ).clip(-8, 4)

    # -------------------------------------------------
    # Prior selection modes
    # -------------------------------------------------
    if prior_mode == "global":
        out["a_idm_prior"] = out["a_idm_global"]

    elif prior_mode == "piecewise_tree":
        if selector is None:
            raise ValueError("selector is required for piecewise_tree")

        out["regime_pred"] = selector.predict(out[selector_feature_cols])
        out["a_idm_prior"] = np.nan

        out.loc[out["regime_pred"] == "braking", "a_idm_prior"] = out.loc[
            out["regime_pred"] == "braking", "a_idm_braking"
        ]

        out.loc[out["regime_pred"] == "accelerating", "a_idm_prior"] = out.loc[
            out["regime_pred"] == "accelerating", "a_idm_accel"
        ]

        out.loc[out["regime_pred"] == "steady", "a_idm_prior"] = out.loc[
            out["regime_pred"] == "steady", "a_idm_steady"
        ]

        out["a_idm_prior"] = out["a_idm_prior"].clip(-8, 4)

    elif prior_mode == "piecewise_gnb":
        if selector is None:
            raise ValueError("selector is required for piecewise_gnb")

        out["regime_pred"] = selector.predict(out[selector_feature_cols])
        out["a_idm_prior"] = np.nan

        out.loc[out["regime_pred"] == "braking", "a_idm_prior"] = out.loc[
            out["regime_pred"] == "braking", "a_idm_braking"
        ]

        out.loc[out["regime_pred"] == "accelerating", "a_idm_prior"] = out.loc[
            out["regime_pred"] == "accelerating", "a_idm_accel"
        ]

        out.loc[out["regime_pred"] == "steady", "a_idm_prior"] = out.loc[
            out["regime_pred"] == "steady", "a_idm_steady"
        ]

        out["a_idm_prior"] = out["a_idm_prior"].clip(-8, 4)

    elif prior_mode == "xgb_best_idm_soft":
        if selector is None:
            raise ValueError("selector bundle is required for xgb_best_idm_soft")

        selector_model = selector["model"]
        label_encoder = selector["label_encoder"]

        proba = selector_model.predict_proba(out[selector_feature_cols])
        classes = list(label_encoder.classes_)

        class_to_col = {
            "global": "a_idm_global",
            "braking": "a_idm_braking",
            "accelerating": "a_idm_accel",
            "steady": "a_idm_steady"
        }

        out["a_idm_prior"] = 0.0

        for j, cls in enumerate(classes):
            out[f"p_{cls}"] = proba[:, j]
            out["a_idm_prior"] += proba[:, j] * out[class_to_col[cls]].values

        hard_idx = np.argmax(proba, axis=1)
        out["xgb_best_idm_hard_label"] = label_encoder.inverse_transform(hard_idx)

        out["a_idm_prior"] = out["a_idm_prior"].clip(-8, 4)

    elif prior_mode == "oracle_piecewise":
        out = add_regime_column(out, true_col="a_true")
        out["a_idm_prior"] = np.nan

        out.loc[out["regime"] == "braking", "a_idm_prior"] = out.loc[
            out["regime"] == "braking", "a_idm_braking"
        ]

        out.loc[out["regime"] == "accelerating", "a_idm_prior"] = out.loc[
            out["regime"] == "accelerating", "a_idm_accel"
        ]

        out.loc[out["regime"] == "steady", "a_idm_prior"] = out.loc[
            out["regime"] == "steady", "a_idm_steady"
        ]

        out["a_idm_prior"] = out["a_idm_prior"].clip(-8, 4)

    elif prior_mode == "subset_braking":
        out["a_idm_prior"] = out["a_idm_braking"]

    elif prior_mode == "subset_accel":
        out["a_idm_prior"] = out["a_idm_accel"]

    elif prior_mode == "subset_steady":
        out["a_idm_prior"] = out["a_idm_steady"]

    else:
        raise ValueError(f"Unknown prior_mode: {prior_mode}")

    missing_prior = out["a_idm_prior"].isna().sum()
    if missing_prior > 0:
        raise ValueError(f"{prior_mode}: missing a_idm_prior for {missing_prior} rows")

    out["residual_true"] = out["a_true"] - out["a_idm_prior"]

    return out

In [32]:
# ============================================================
# SEQUENCE SETTINGS AND BUILDERS
# ============================================================

# Use the same history length as the Transformer baseline.
# STRIDE = 1 means every possible sequence target is used.
HISTORY = 25
STRIDE = 1

SEQUENCE_TAG = f"H{HISTORY}_S{STRIDE}"

BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-3

model_feature_cols = [
    "v_ego",
    "v_leader",
    "delta_v",
    "gap",
    "a_idm_prior"
]


def build_sequences(df, history=HISTORY, stride=STRIDE):
    sequences_X = []
    targets_y = []
    a_idm_targets = []
    a_true_targets = []
    meta_rows = []

    for episode_id, g in df.groupby("episode_id", sort=False):
        g = g.sort_values("frame").reset_index(drop=True)

        X = g[model_feature_cols].to_numpy(dtype=np.float32)
        y = g["residual_true"].to_numpy(dtype=np.float32)
        a_idm = g["a_idm_prior"].to_numpy(dtype=np.float32)
        a_true = g["a_true"].to_numpy(dtype=np.float32)

        for i in range(history - 1, len(g), stride):
            start = i - history + 1
            end = i + 1

            sequences_X.append(X[start:end])
            targets_y.append(y[i])
            a_idm_targets.append(a_idm[i])
            a_true_targets.append(a_true[i])

            row = {
                "episode_id": g.loc[i, "episode_id"],
                "unique_id": g.loc[i, "unique_id"],
                "frame": g.loc[i, "frame"],
                "a_true": g.loc[i, "a_true"],
                "a_idm_prior": g.loc[i, "a_idm_prior"],
            }

            for col in [
                "regime",
                "regime_pred",
                "xgb_best_idm_hard_label",
                "p_global",
                "p_braking",
                "p_accelerating",
                "p_steady",
                "a_idm_global",
                "a_idm_braking",
                "a_idm_accel",
                "a_idm_steady",
            ]:
                if col in g.columns:
                    row[col] = g.loc[i, col]

            meta_rows.append(row)

    X_arr = np.asarray(sequences_X, dtype=np.float32)
    y_arr = np.asarray(targets_y, dtype=np.float32)
    a_idm_arr = np.asarray(a_idm_targets, dtype=np.float32)
    a_true_arr = np.asarray(a_true_targets, dtype=np.float32)
    meta_df = pd.DataFrame(meta_rows)

    assert len(X_arr) == len(y_arr) == len(a_idm_arr) == len(a_true_arr) == len(meta_df), (
        f"Length mismatch: X={len(X_arr)}, y={len(y_arr)}, "
        f"a_idm={len(a_idm_arr)}, a_true={len(a_true_arr)}, meta={len(meta_df)}"
    )

    return X_arr, y_arr, a_idm_arr, a_true_arr, meta_df


def fit_sequence_scalers(X_train, y_train):
    feature_mean = X_train.reshape(-1, X_train.shape[-1]).mean(axis=0)
    feature_std = X_train.reshape(-1, X_train.shape[-1]).std(axis=0)
    feature_std = np.where(feature_std < 1e-8, 1.0, feature_std)

    target_mean = y_train.mean()
    target_std = y_train.std()

    if target_std < 1e-8:
        target_std = 1.0

    return {
        "feature_mean": feature_mean,
        "feature_std": feature_std,
        "target_mean": target_mean,
        "target_std": target_std
    }


def transform_sequences(X, y, scalers):
    X_scaled = (X - scalers["feature_mean"]) / scalers["feature_std"]
    y_scaled = (y - scalers["target_mean"]) / scalers["target_std"]

    return X_scaled.astype(np.float32), y_scaled.astype(np.float32)


def inverse_transform_target(y_scaled, scalers):
    return y_scaled * scalers["target_std"] + scalers["target_mean"]


print("Sequence settings:")
print("HISTORY:", HISTORY)
print("STRIDE:", STRIDE)
print("SEQUENCE_TAG:", SEQUENCE_TAG)

Sequence settings:
HISTORY: 25
STRIDE: 1
SEQUENCE_TAG: H25_S1


In [33]:
# ============================================================
# SAVE / LOAD HELPERS FOR RESUMABLE RUNS
# ============================================================

def save_prior_dataframes(run_dir, train_prior_df, val_prior_df, test_prior_df):
    train_prior_df.to_csv(run_dir / "train_prior.csv", index=False)
    val_prior_df.to_csv(run_dir / "val_prior.csv", index=False)
    test_prior_df.to_csv(run_dir / "test_prior.csv", index=False)


def load_prior_dataframes(run_dir):
    train_prior_df = pd.read_csv(run_dir / "train_prior.csv")
    val_prior_df = pd.read_csv(run_dir / "val_prior.csv")
    test_prior_df = pd.read_csv(run_dir / "test_prior.csv")

    return train_prior_df, val_prior_df, test_prior_df


def save_sequence_bundle(path_npz, X, y, a_idm_target, a_true_target, meta_df):
    np.savez_compressed(
        path_npz,
        X=X,
        y=y,
        a_idm_target=a_idm_target,
        a_true_target=a_true_target,
    )

    meta_df.to_csv(
        Path(path_npz).with_suffix(".meta.csv"),
        index=False
    )


def load_sequence_bundle(path_npz):
    path_npz = Path(path_npz)
    data = np.load(path_npz, allow_pickle=True)

    X = data["X"]
    y = data["y"]
    a_idm_target = data["a_idm_target"]
    a_true_target = data["a_true_target"]

    meta_path = path_npz.with_suffix(".meta.csv")
    meta_df = pd.read_csv(meta_path)

    return X, y, a_idm_target, a_true_target, meta_df


def save_scalers(run_dir, scalers):
    np.savez_compressed(
        run_dir / "scalers.npz",
        feature_mean=scalers["feature_mean"],
        feature_std=scalers["feature_std"],
        target_mean=scalers["target_mean"],
        target_std=scalers["target_std"]
    )


def load_scalers(run_dir):
    data = np.load(run_dir / "scalers.npz", allow_pickle=True)

    return {
        "feature_mean": data["feature_mean"],
        "feature_std": data["feature_std"],
        "target_mean": float(data["target_mean"]),
        "target_std": float(data["target_std"]),
    }


def save_training_log(run_dir, log_rows):
    pd.DataFrame(log_rows).to_csv(run_dir / "train_log.csv", index=False)

In [34]:
class HybridSequenceDataset(Dataset):
    def __init__(self, X, y, a_idm_target, a_true_target, meta_df):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.a_idm_target = torch.tensor(a_idm_target, dtype=torch.float32)
        self.a_true_target = torch.tensor(a_true_target, dtype=torch.float32)
        self.meta_df = meta_df.reset_index(drop=True)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return {
            "X": self.X[idx],
            "y": self.y[idx],
            "a_idm_target": self.a_idm_target[idx],
            "a_true_target": self.a_true_target[idx],
            "idx": idx
        }

In [35]:
class HybridTransformer(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        h = self.input_proj(x)
        h = self.encoder(h)
        h_last = h[:, -1, :]
        out = self.head(h_last).squeeze(-1)
        return out

In [36]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for batch in loader:
        X = batch["X"].to(DEVICE)
        y = batch["y"].to(DEVICE)

        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(X)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict_residuals(model, loader):
    model.eval()

    preds = []
    ys = []
    a_idm_targets = []
    a_true_targets = []
    indices = []

    for batch in loader:
        X = batch["X"].to(DEVICE)
        pred = model(X).cpu().numpy()

        preds.append(pred)
        ys.append(batch["y"].cpu().numpy())
        a_idm_targets.append(batch["a_idm_target"].cpu().numpy())
        a_true_targets.append(batch["a_true_target"].cpu().numpy())
        indices.append(batch["idx"].cpu().numpy())

    return {
        "pred_scaled": np.concatenate(preds),
        "y_scaled": np.concatenate(ys),
        "a_idm_target": np.concatenate(a_idm_targets),
        "a_true_target": np.concatenate(a_true_targets),
        "idx": np.concatenate(indices)
    }


def evaluate_hybrid_predictions(pred_dict, scalers, meta_df):
    residual_pred = inverse_transform_target(pred_dict["pred_scaled"], scalers)
    residual_true = inverse_transform_target(pred_dict["y_scaled"], scalers)

    a_idm_target = pred_dict["a_idm_target"]
    a_true_target = pred_dict["a_true_target"]

    a_hybrid = a_idm_target + residual_pred

    metrics = compute_metrics(a_true_target, a_hybrid)

    out_df = meta_df.copy().reset_index(drop=True)
    out_df["residual_true"] = residual_true
    out_df["residual_pred"] = residual_pred
    out_df["a_idm_target"] = a_idm_target
    out_df["a_true_target"] = a_true_target
    out_df["a_hybrid"] = a_hybrid
    out_df["y_true"] = a_true_target
    out_df["y_pred"] = a_hybrid
    out_df["error"] = out_df["y_pred"] - out_df["y_true"]
    out_df["abs_error"] = out_df["error"].abs()
    out_df["Model"] = "Hybrid"

    return out_df, metrics

In [37]:
def run_hybrid_experiment(
    name,
    train_base_df,
    val_base_df,
    test_base_df,
    prior_mode,
    selector=None,
    run_dir=None
):
    if run_dir is None:
        raise ValueError("run_dir must be provided")

    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------------------------------
    # PATHS
    # -------------------------------------------------
    train_prior_path = run_dir / "train_prior.csv"
    val_prior_path   = run_dir / "val_prior.csv"
    test_prior_path  = run_dir / "test_prior.csv"

    train_seq_path = run_dir / "train_sequences.npz"
    val_seq_path   = run_dir / "val_sequences.npz"
    test_seq_path  = run_dir / "test_sequences.npz"

    scalers_path = run_dir / "scalers.npz"
    best_model_path = run_dir / "best_model.pt"
    last_checkpoint_path = run_dir / "last_checkpoint.pt"
    checkpoints_dir = run_dir / "checkpoints"
    checkpoints_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------------------------------
    # BUILD OR LOAD PRIOR DATAFRAMES
    # -------------------------------------------------
    if train_prior_path.exists() and val_prior_path.exists() and test_prior_path.exists():
        print(f"{name} | loading saved prior dataframes")
        train_prior_df, val_prior_df, test_prior_df = load_prior_dataframes(run_dir)
    else:
        print(f"{name} | building prior dataframes")
        train_prior_df = build_prior_dataframe(train_base_df, prior_mode=prior_mode, selector=selector)
        val_prior_df   = build_prior_dataframe(val_base_df, prior_mode=prior_mode, selector=selector)
        test_prior_df  = build_prior_dataframe(test_base_df, prior_mode=prior_mode, selector=selector)

        save_prior_dataframes(run_dir, train_prior_df, val_prior_df, test_prior_df)
        print(f"{name} | saved prior dataframes")

    # -------------------------------------------------
    # BUILD OR LOAD SEQUENCES
    # -------------------------------------------------
    if train_seq_path.exists() and val_seq_path.exists() and test_seq_path.exists():
        print(f"{name} | loading saved sequences")
        X_train, y_train, a_idm_train, a_true_train, meta_train = load_sequence_bundle(train_seq_path)
        X_val, y_val, a_idm_val, a_true_val, meta_val = load_sequence_bundle(val_seq_path)
        X_test, y_test, a_idm_test, a_true_test, meta_test = load_sequence_bundle(test_seq_path)
    else:
        print(f"{name} | building sequences")
        X_train, y_train, a_idm_train, a_true_train, meta_train = build_sequences(train_prior_df)
        X_val, y_val, a_idm_val, a_true_val, meta_val = build_sequences(val_prior_df)
        X_test, y_test, a_idm_test, a_true_test, meta_test = build_sequences(test_prior_df)

        save_sequence_bundle(train_seq_path, X_train, y_train, a_idm_train, a_true_train, meta_train)
        save_sequence_bundle(val_seq_path, X_val, y_val, a_idm_val, a_true_val, meta_val)
        save_sequence_bundle(test_seq_path, X_test, y_test, a_idm_test, a_true_test, meta_test)
        print(f"{name} | saved sequences")

    # -------------------------------------------------
    # FIT OR LOAD SCALERS
    # -------------------------------------------------
    if scalers_path.exists():
        print(f"{name} | loading scalers")
        scalers = load_scalers(run_dir)
    else:
        print(f"{name} | fitting scalers")
        scalers = fit_sequence_scalers(X_train, y_train)
        save_scalers(run_dir, scalers)
        print(f"{name} | saved scalers")

    # -------------------------------------------------
    # TRANSFORM
    # -------------------------------------------------
    X_train_s, y_train_s = transform_sequences(X_train, y_train, scalers)
    X_val_s, y_val_s = transform_sequences(X_val, y_val, scalers)
    X_test_s, y_test_s = transform_sequences(X_test, y_test, scalers)

    # -------------------------------------------------
    # DATASETS / LOADERS
    # -------------------------------------------------
    train_ds = HybridSequenceDataset(X_train_s, y_train_s, a_idm_train, a_true_train, meta_train)
    val_ds   = HybridSequenceDataset(X_val_s, y_val_s, a_idm_val, a_true_val, meta_val)
    test_ds  = HybridSequenceDataset(X_test_s, y_test_s, a_idm_test, a_true_test, meta_test)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    # -------------------------------------------------
    # MODEL
    # -------------------------------------------------
    model = HybridTransformer(input_dim=len(model_feature_cols)).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()

    best_val_rmse = np.inf
    best_model_state = None
    best_epoch = -1
    log_rows = []
    start_epoch = 0

    # resume logic
    if last_checkpoint_path.exists():
        print(f"{name} | resuming from checkpoint")
        ckpt = torch.load(last_checkpoint_path, map_location=DEVICE, weights_only=False)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        best_val_rmse = ckpt["best_val_rmse"]
        best_epoch = ckpt["best_epoch"]
        log_rows = ckpt["log_rows"]
        start_epoch = ckpt["epoch"] + 1

        best_model_state = torch.load(best_model_path, map_location=DEVICE, weights_only=False)

    # -------------------------------------------------
    # TRAIN
    # -------------------------------------------------
    if best_model_path.exists() and start_epoch >= EPOCHS:
        print(f"{name} | training already completed, loading best model")
        best_model_state = torch.load(best_model_path, map_location=DEVICE)
    else:
        for epoch in range(start_epoch, EPOCHS):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
            val_pred_dict = predict_residuals(model, val_loader)
            _, val_metrics = evaluate_hybrid_predictions(val_pred_dict, scalers, meta_val)

            val_rmse = val_metrics["RMSE"]

            if val_rmse < best_val_rmse:
                best_val_rmse = val_rmse
                best_epoch = epoch + 1
                best_model_state = copy.deepcopy(model.state_dict())
                torch.save(best_model_state, best_model_path)

            log_rows.append({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "val_rmse": val_rmse
            })
            save_training_log(run_dir, log_rows)

            ckpt_payload = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_rmse": best_val_rmse,
                "best_epoch": best_epoch,
                "log_rows": log_rows
            }

            torch.save(ckpt_payload, last_checkpoint_path)
            torch.save(ckpt_payload, checkpoints_dir / f"checkpoint_epoch_{epoch+1:02d}.pt")

            print(
                f"{name} | epoch {epoch+1}/{EPOCHS} | "
                f"train_loss={train_loss:.6f} | val_RMSE={val_rmse:.6f}"
            )

    # -------------------------------------------------
    # LOAD BEST MODEL
    # -------------------------------------------------
    if best_model_state is None:
        raise RuntimeError(f"{name} | best_model_state is None")

    model.load_state_dict(best_model_state)

    # -------------------------------------------------
    # FINAL PREDICTIONS
    # -------------------------------------------------
    train_pred_dict = predict_residuals(model, train_loader)
    val_pred_dict   = predict_residuals(model, val_loader)
    test_pred_dict  = predict_residuals(model, test_loader)

    train_out_df, train_metrics = evaluate_hybrid_predictions(train_pred_dict, scalers, meta_train)
    val_out_df, val_metrics     = evaluate_hybrid_predictions(val_pred_dict, scalers, meta_val)
    test_out_df, test_metrics   = evaluate_hybrid_predictions(test_pred_dict, scalers, meta_test)

    regime_metrics_test = compute_regime_metrics_from_arrays(
        y_true=test_out_df["a_true_target"].values,
        y_pred=test_out_df["a_hybrid"].values
    )

    # save final outputs
    train_out_df.to_csv(run_dir / "train_predictions.csv", index=False)
    val_out_df.to_csv(run_dir / "val_predictions.csv", index=False)
    test_out_df.to_csv(run_dir / "test_predictions.csv", index=False)

    pd.DataFrame([{"split": "train", **train_metrics}]).to_csv(run_dir / "train_metrics.csv", index=False)
    pd.DataFrame([{"split": "val", **val_metrics}]).to_csv(run_dir / "val_metrics.csv", index=False)
    pd.DataFrame([{"split": "test", **test_metrics}]).to_csv(run_dir / "test_metrics.csv", index=False)
    regime_metrics_test.to_csv(run_dir / "test_regime_metrics.csv", index=False)

    return {
        "name": name,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "test_predictions": test_out_df,
        "test_regime_metrics": regime_metrics_test
    }

In [38]:
# ============================================================
# FINAL IDM-FIRST HYBRID EXPERIMENT SUITE
# Runs all final experiments and saves each one immediately.
# Safe for overheating/interruption: completed runs are reused.
# ============================================================

FINAL_EXPERIMENTS = [
    {
        "name": "Hybrid_GlobalIDM",
        "prior_mode": "global",
        "selector": None,
        "description": "Residual hybrid using global IDM prior."
    },
    {
        "name": "Hybrid_TreeRegimeAwareIDM",
        "prior_mode": "piecewise_tree",
        "selector": selector_tree,
        "description": "Residual hybrid using tree-selected regime-aware IDM prior."
    },
    {
        "name": "Hybrid_XGBSoftBestIDM",
        "prior_mode": "xgb_best_idm_soft",
        "selector": xgb_best_selector_bundle,
        "description": "Residual hybrid using XGB-soft best-IDM prior."
    },
    {
        "name": "Hybrid_OracleRegimeAwareIDM",
        "prior_mode": "oracle_piecewise",
        "selector": None,
        "description": "Residual hybrid using true-regime oracle IDM prior."
    },
]

SKIP_COMPLETED = True

suite_rows = []
suite_regime_rows = []

for exp in FINAL_EXPERIMENTS:
    experiment_name = exp["name"]
    prior_mode = exp["prior_mode"]
    selector = exp["selector"]

    run_dir = OUTPUT_DIR / f"{experiment_name}_{SEQUENCE_TAG}_{THRESHOLD_TAG}_{IDM_PARAM_TAG}"
    run_dir.mkdir(parents=True, exist_ok=True)

    metrics_path = run_dir / "test_metrics.csv"
    regime_path = run_dir / "test_regime_metrics.csv"
    predictions_path = run_dir / "test_predictions.csv"
    config_path = run_dir / "config.json"

    print("\n" + "=" * 80)
    print("Experiment:", experiment_name)
    print("Prior mode:", prior_mode)
    print("Run dir:", run_dir)
    print("=" * 80)

    if SKIP_COMPLETED and metrics_path.exists() and regime_path.exists() and predictions_path.exists():
        print("Completed outputs found. Loading existing results instead of rerunning.")

        metrics_df_existing = pd.read_csv(metrics_path)
        regime_df_existing = pd.read_csv(regime_path)

        metrics_row_existing = metrics_df_existing.iloc[0].to_dict()

        # Make old saved files compatible with the final suite table
        metrics_row_existing["Experiment"] = experiment_name
        metrics_row_existing["PriorMode"] = prior_mode
        metrics_row_existing["Description"] = exp["description"]

        # Remove possible old column from earlier versions
        metrics_row_existing.pop("split", None)

        suite_rows.append(metrics_row_existing)

        temp_regime = regime_df_existing.copy()
        temp_regime["Experiment"] = experiment_name
        temp_regime["PriorMode"] = prior_mode

        suite_regime_rows.append(temp_regime)

        continue

    result = run_hybrid_experiment(
        name=experiment_name,
        train_base_df=train_df_raw,
        val_base_df=val_df_raw,
        test_base_df=test_df_raw,
        prior_mode=prior_mode,
        selector=selector,
        run_dir=run_dir
    )

    # -----------------------------
    # Save immediately after run
    # -----------------------------
    metrics_row = {
        "Experiment": result["name"],
        "PriorMode": prior_mode,
        "Description": exp["description"],
        **result["test_metrics"]
    }

    metrics_df = pd.DataFrame([metrics_row])
    metrics_df.to_csv(metrics_path, index=False)

    regime_df = result["test_regime_metrics"].copy()
    regime_df["Experiment"] = result["name"]
    regime_df["PriorMode"] = prior_mode
    regime_df.to_csv(regime_path, index=False)

    test_predictions = result["test_predictions"].copy()
    test_predictions["Model"] = experiment_name
    test_predictions["PriorMode"] = prior_mode
    test_predictions.to_csv(predictions_path, index=False)

    with open(config_path, "w") as f:
        json.dump({
            "EXPERIMENT_NAME": experiment_name,
            "PRIOR_MODE": prior_mode,
            "DESCRIPTION": exp["description"],
            "BRAKE_THRESHOLD": BRAKE_THRESHOLD,
            "ACCEL_THRESHOLD": ACCEL_THRESHOLD,
            "THRESHOLD_TAG": THRESHOLD_TAG,
            "HISTORY": HISTORY,
            "STRIDE": STRIDE,
            "SEQUENCE_TAG": SEQUENCE_TAG,
            "BATCH_SIZE": BATCH_SIZE,
            "EPOCHS": EPOCHS,
            "LR": LR,
            "model_feature_cols": model_feature_cols,
            "selector_feature_cols": selector_feature_cols,
            "DATA_DIR": str(DATA_DIR),
            "RUN_DIR": str(run_dir)
        }, f, indent=2)

    print("Saved:")
    print(metrics_path)
    print(regime_path)
    print(predictions_path)
    print(config_path)

    suite_rows.append(metrics_row)
    suite_regime_rows.append(regime_df)

# ============================================================
# SUITE SUMMARY
# ============================================================

suite_summary_df = pd.DataFrame(suite_rows)

required_metric_cols = ["Experiment", "PriorMode", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
missing_cols = [c for c in required_metric_cols if c not in suite_summary_df.columns]

if missing_cols:
    raise ValueError(f"Suite summary is missing columns: {missing_cols}")

suite_summary_df = suite_summary_df.sort_values("MAE").reset_index(drop=True)

suite_regime_df = pd.concat(suite_regime_rows, ignore_index=True)

suite_summary_path = OUTPUT_DIR / f"hybrid_idm_first_suite_summary_{SEQUENCE_TAG}_{THRESHOLD_TAG}.csv"
suite_regime_path = OUTPUT_DIR / f"hybrid_idm_first_suite_regime_metrics_{SEQUENCE_TAG}_{THRESHOLD_TAG}.csv"

suite_summary_df.to_csv(suite_summary_path, index=False)
suite_regime_df.to_csv(suite_regime_path, index=False)

print("\n" + "=" * 80)
print("FINAL IDM-FIRST HYBRID SUITE SUMMARY")
print("=" * 80)
print(
    suite_summary_df[
        ["Experiment", "PriorMode", "MAE", "RMSE", "MedAE", "R2", "Bias", "Corr"]
    ].to_string(index=False)
)

print("\nSaved suite summary:")
print(suite_summary_path)
print(suite_regime_path)


Experiment: Hybrid_GlobalIDM
Prior mode: global
Run dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\hybrid_idm_first_runs\Hybrid_GlobalIDM_H25_S1_brake0p2_accel0p2_refinedIDM_v2
Completed outputs found. Loading existing results instead of rerunning.

Experiment: Hybrid_TreeRegimeAwareIDM
Prior mode: piecewise_tree
Run dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\hybrid_idm_first_runs\Hybrid_TreeRegimeAwareIDM_H25_S1_brake0p2_accel0p2_refinedIDM_v2
Completed outputs found. Loading existing results instead of rerunning.

Experiment: Hybrid_XGBSoftBestIDM
Prior mode: xgb_best_idm_soft
Run dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\hybrid_idm_first_runs\Hybrid_XGBSoftBestIDM_H25_S1_brake0p2_accel0p2_refinedIDM_v2
Hybrid_XGBSoftBestIDM | loading saved prior dataframes
Hybrid_XGBSoftBestIDM | loading saved sequences
Hybrid_XGBSoftBestIDM | loading scalers
Hybrid_XGBSoftBestIDM | 